# Sumativa 3: Preparación e imputación del ingreso mensual

**Asignatura:** Estadística Computacional  
**Integrantes:** Hernán Saavedra, Lenin Morales y Rodrigo Gómez  
**Responsable de este notebook:** Hernán Saavedra  
**Dataset:** IBM HR Employee Attrition

## Objetivo

Comparar eliminación de registros, imputación por mediana e imputación
mediante regresión lineal múltiple para preparar los datos del
modelamiento logístico final.

## Continuidad con S1, S2 y Formativa 2

El dataset original utilizado en S1 y S2 no presentó valores faltantes.
Siguiendo la indicación docente, en Formativa 2 se ocultaron 147 valores
de MonthlyIncome (10 %) mediante un mecanismo MCAR con semilla 42.

Se reutilizan esa versión, el respaldo de los valores ocultados y la
partición estratificada por Attrition: 1.029 registros de entrenamiento
y 441 de prueba. No se generan nuevos faltantes ni una nueva partición.

Las correlaciones y los análisis de robustez de S1 y S2 fundamentarán
la elección de predictores para imputar el ingreso. Los modelos de
imputación se ajustarán exclusivamente con entrenamiento, sin utilizar
Attrition como predictor.

Los valores ocultados se reservarán para evaluar el error de imputación.
Los datos son sintéticos y las conclusiones tienen finalidad académica.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.impute import SimpleImputer

SEMILLA = 42

RAIZ = next(
    (
        carpeta
        for carpeta in [Path.cwd(), *Path.cwd().parents]
        if (
            carpeta / "datos" / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
        ).is_file()
    ),
    None,
)

if RAIZ is None:
    raise FileNotFoundError("No se encontró la raíz del repositorio.")

DIRECTORIO_F2 = RAIZ / "Formativa2"
DIRECTORIO_S2 = RAIZ / "S2"
DIRECTORIO_S3 = RAIZ / "S3"

for nombre in ["datos", "resultados", "figuras", "docs"]:
    (DIRECTORIO_S3 / nombre).mkdir(parents=True, exist_ok=True)

archivos_requeridos = [
    DIRECTORIO_F2 / "datos" / "IBM_HR_MCAR.csv",
    DIRECTORIO_F2 / "datos" / "valores_ocultados_ingreso.csv",
    DIRECTORIO_F2 / "datos" / "particion_70_30.csv",
    DIRECTORIO_S2 / "resultados" / "S2_reporte_validado_S3.json",
]

for ruta in archivos_requeridos:
    if not ruta.is_file():
        raise FileNotFoundError(
            f"Falta un insumo previo: {ruta.relative_to(RAIZ)}"
        )

print("Python:", sys.version.split()[0])
print("Insumos de F2 y S2 encontrados.")
print("Entorno preparado para Sumativa 3.")

Python: 3.14.2
Insumos de F2 y S2 encontrados.
Entorno preparado para Sumativa 3.


## 1. Recuperación de datos y partición de Formativa 2

Se carga la versión MCAR guardada en F2 y se comprueba que únicamente
MonthlyIncome presenta faltantes. Se verifican los identificadores y
el respaldo contra el dataset original.

La asignación a entrenamiento y prueba se recupera del archivo de
partición. El dataset original y los valores ocultados se mantienen
separados de los datos empleados para ajustar la imputación.

In [2]:
df_original = pd.read_csv(
    RAIZ / "datos" / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
).set_index("EmployeeNumber")

df_mcar = pd.read_csv(
    DIRECTORIO_F2 / "datos" / "IBM_HR_MCAR.csv"
).set_index("EmployeeNumber")

valores_ocultados = pd.read_csv(
    DIRECTORIO_F2 / "datos" / "valores_ocultados_ingreso.csv"
).set_index("EmployeeNumber")

particion = pd.read_csv(
    DIRECTORIO_F2 / "datos" / "particion_70_30.csv"
).set_index("EmployeeNumber")

reporte_s2 = json.loads(
    (
        DIRECTORIO_S2 / "resultados" / "S2_reporte_validado_S3.json"
    ).read_text(encoding="utf-8")
)

# Verificar que se conserva la versión MCAR de F2.
assert df_mcar.index.is_unique
assert valores_ocultados.index.is_unique
assert particion.index.is_unique
assert set(particion.index) == set(df_mcar.index)
assert set(particion["Conjunto"]) == {"Entrenamiento", "Prueba"}

assert df_mcar["MonthlyIncome"].isna().sum() == 147
assert df_mcar.isna().sum().sum() == 147

ids_ocultados = df_mcar.index[df_mcar["MonthlyIncome"].isna()]
assert set(ids_ocultados) == set(valores_ocultados.index)

pd.testing.assert_frame_equal(
    df_mcar.drop(columns="MonthlyIncome"),
    df_original.drop(columns="MonthlyIncome"),
)

np.testing.assert_allclose(
    valores_ocultados["MonthlyIncome"],
    df_original.loc[valores_ocultados.index, "MonthlyIncome"],
)

# Recuperar la asignación existente, sin sortear otra partición.
train = df_mcar.loc[
    particion.index[particion["Conjunto"] == "Entrenamiento"]
].copy()

test = df_mcar.loc[
    particion.index[particion["Conjunto"] == "Prueba"]
].copy()

assert len(train) == 1029 and len(test) == 441
assert set(train.index).isdisjoint(test.index)

resumen_datos = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Prueba"],
    "Registros": [len(train), len(test)],
    "Ingresos faltantes": [
        train["MonthlyIncome"].isna().sum(),
        test["MonthlyIncome"].isna().sum(),
    ],
    "Salidas": [
        train["Attrition"].eq("Yes").sum(),
        test["Attrition"].eq("Yes").sum(),
    ],
})

display(resumen_datos)
print("Partición de F2 recuperada y respaldo verificado.")
print("Reporte de resultados validados de S2 cargado.")

,Conjunto,Registros,Ingresos faltantes,Salidas
0,Entrenamiento,1029,96,166
1,Prueba,441,51,71


Partición de F2 recuperada y respaldo verificado.
Reporte de resultados validados de S2 cargado.
